#### Q. how langchain uses mcp servers?
- langchain uses `from langchain_mcp_adapters.client import MultiServerMCPClient` to instantiate mcp client
- then it get the tools for that mcp client
- bind the llm to those tools

In [1]:
import asyncio
from typing import Annotated, TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

/home/alin/miniconda3/envs/langchain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

### using `playwright`

In [3]:

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def route(s: State):
    return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

async def main():
    mcp = MultiServerMCPClient({
        "playwright": {"transport": "stdio", "command": "npx", "args": ["@playwright/mcp@latest"]}
    })
    tools = await mcp.get_tools()
    llm = ChatOpenAI(model="gpt-4.1-mini").bind_tools(tools)

    async def chat(s: State):
        return {"messages": [await llm.ainvoke(s["messages"])]}

    g = StateGraph(State)
    g.add_node("chat", chat)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "chat")
    g.add_conditional_edges("chat", route, {"tools": "tools", END: END})
    g.add_edge("tools", "chat")

    app = g.compile()
    # human_message = "search www.canadiantire.ca for Ninja Foodi 2-Basket Air Fryer and find the current price"
    human_message = "what is temperature in vancouver right now?"

    out = await app.ainvoke({"messages": [HumanMessage(content=human_message)]} )
    print(out["messages"][-1].content)

await main()


I attempted to load the weather information for Vancouver from weather.com, but the page did not load correctly and returned a 404 error. Therefore, I couldn't retrieve the current temperature for Vancouver from that source.

Would you like me to try another source or method to find the current temperature in Vancouver?


### using `serpapi`

In [4]:
import os
from typing import Annotated, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

In [7]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def route(s: State):
    return "tools" if getattr(s["messages"][-1], "tool_calls", None) else END

async def main():
    key = os.environ["SERPAPI_API_KEY"]
    mcp = MultiServerMCPClient({
        "serpapi": {"transport": "http", "url": f"https://mcp.serpapi.com/{key}/mcp"}
    })
    tools = await mcp.get_tools()
    llm = ChatOpenAI(model="gpt-4.1-mini").bind_tools(tools)

    async def chat(s: State): return {"messages": [await llm.ainvoke(s["messages"])]}

    g = StateGraph(State)
    g.add_node("chat", chat)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "chat")
    g.add_conditional_edges("chat", route, {"tools": "tools", END: END})
    g.add_edge("tools", "chat")

    human_message = "search amazon.ca for Ninja Foodi 2-Basket Air Fryer and find the current price"
    # human_message = "search google for current weather in Calgary, Alberta"
    out = await g.compile().ainvoke({
        "messages": [HumanMessage(content=human_message)]
    })
    print(out["messages"][-1].content)


await main()

I couldn't find current price information for the Ninja Foodi 2-Basket Air Fryer specifically on amazon.ca through the shopping search. Would you like me to try a broader search or check a different source for this product?
